In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent

SYNTHETIC_DATA_DIR = REPO_ROOT / 'data' / 'synthetic'
RAW_DATA_DIR = REPO_ROOT / 'data' / 'raw'


## Pendulum synthetic data
This notebook simulates an pendulum swinging in the x-z plane (no-damping) and saves 5 trials of noisy IMU measurements (accelerometer + gyroscope) with ground-truth orientation and velocity for use in EKF/Madgwick development. Run this notebook first to generate the synthetic data required by all subsequent notebooks.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# Constants
G  = 9.81
DT = 1 / 100    # 100 Hz

# Noise Parameters 
SIGMA_ACCEL      = 0.05    # m/s²      accelerometer white noise std
SIGMA_GYRO       = 0.01    # rad/s     gyroscope white noise std
SIGMA_BIAS_ACCEL = 0.001   # m/s²/√s  accelerometer random-walk bias
SIGMA_BIAS_GYRO  = 0.0001  # rad/s/√s gyroscope random-walk bias

# Trial Settings (5 trials with different random seeds)
SEEDS    = [42, 7, 123, 256, 999]
N_TRIALS = len(SEEDS)


In [2]:
def generate_pendulum_data(
    N: int,
    L: float = 1.0,
    initial_angle: float = np.pi / 4,
    initial_angular_velocity: float = 0.0,
) -> tuple:
    """
    Generates synthetic IMU measurements and ground truth for a simple pendulum.

    Parameters
    ----------
    N                        : number of samples
    L                        : pendulum length (m)
    initial_angle            : initial angle from vertical (rad)
    initial_angular_velocity : initial angular velocity (rad/s)

    Returns
    -------
    Z  : (N, 6) float64  noisy IMU measurements [ax, ay, az, wx, wy, wz]
    gt : dict  with keys 't', 'roll', 'pitch', 'yaw', 'vx', 'vy', 'vz'
    """
    t = np.arange(N) * DT

    # initialize ground truth states
    gt = {
        "t":     t,
        "roll":  np.zeros(N),
        "pitch": np.zeros(N),
        "yaw":   np.zeros(N),
        "vx":    np.zeros(N),
        "vy":    np.zeros(N),
        "vz":    np.zeros(N),
    }

    angles                = np.zeros(N)
    angular_velocities    = np.zeros(N)
    angular_accelerations = np.zeros(N)

    theta = initial_angle
    omega = initial_angular_velocity

    ax_body_temp = np.zeros(N)
    az_body_temp = np.zeros(N)

    bias   = np.zeros(6)
    biases = np.zeros((N, 6))

    for i in range(N):
        # Pendulum dynamics 
        alpha = -G / L * np.sin(theta) # angular acceleration

        angles[i]                = theta
        angular_velocities[i]    = omega
        angular_accelerations[i] = alpha
        
        # Euler integration
        omega += alpha * DT
        theta += omega * DT

        # ideal body-frame measurements (without noise or bias)
        ax_body_temp[i] = -L * alpha
        az_body_temp[i] =  L * omega**2

        gt["roll"][i]  = 0.0 # should not have roll or yaw for pendulum in x-z plane
        gt["pitch"][i] = angles[i]
        gt["yaw"][i]   = 0.0

        gt["vx"][i] = L * np.cos(angles[i]) * angular_velocities[i] 
        gt["vy"][i] = 0.0 # should not have velocity in y for pendulum in x-z plane
        gt["vz"][i] = L * np.sin(angles[i]) * angular_velocities[i]

        # Define bias as a Wiener process
        bias[0:3] += SIGMA_BIAS_ACCEL * np.sqrt(DT) * np.random.normal(0, 1, 3)
        bias[3:6] += SIGMA_BIAS_GYRO  * np.sqrt(DT) * np.random.normal(0, 1, 3)
        biases[i]  = bias.copy()

    Z = np.zeros((N, 6)) # Ideal noiseless IMU measurements
    Z[:, 0] =  ax_body_temp + G * np.sin(angles)  # tangential acceleration + gravity component on x-axis
    Z[:, 2] = -az_body_temp + G * np.cos(angles)  # centripetal acceleration + gravity component on z-axis
    Z[:, 4] =  angular_velocities

    # Add white noise and bias to measurements
    Z[:, 0] += np.random.normal(0, SIGMA_ACCEL, N) + biases[:, 0]
    Z[:, 1] += np.random.normal(0, SIGMA_ACCEL, N) + biases[:, 1]
    Z[:, 2] += np.random.normal(0, SIGMA_ACCEL, N) + biases[:, 2]
    Z[:, 3] += np.random.normal(0, SIGMA_GYRO,  N) + biases[:, 3]
    Z[:, 4] += np.random.normal(0, SIGMA_GYRO,  N) + biases[:, 4]
    Z[:, 5] += np.random.normal(0, SIGMA_GYRO,  N) + biases[:, 5]

    return Z, gt


In [3]:
# generate and save data for each trial 
for seed in SEEDS:
    np.random.seed(seed)
    Z, gt = generate_pendulum_data(N=1000, L=1.0, initial_angle=np.pi / 6)
    path = SYNTHETIC_DATA_DIR / f'synthetic_data_pendulum_seed{seed}.npz'
    np.savez(path, Z=Z, gt=gt)
    print(f"Saved {path}  |  Z shape: {Z.shape}  |  pitch range: [{gt['pitch'].min():.3f}, {gt['pitch'].max():.3f}] rad")


Saved synthetic_data_pendulum_seed42.npz  |  Z shape: (1000, 6)  |  pitch range: [-0.524, 0.524] rad
Saved synthetic_data_pendulum_seed7.npz  |  Z shape: (1000, 6)  |  pitch range: [-0.524, 0.524] rad
Saved synthetic_data_pendulum_seed123.npz  |  Z shape: (1000, 6)  |  pitch range: [-0.524, 0.524] rad
Saved synthetic_data_pendulum_seed256.npz  |  Z shape: (1000, 6)  |  pitch range: [-0.524, 0.524] rad
Saved synthetic_data_pendulum_seed999.npz  |  Z shape: (1000, 6)  |  pitch range: [-0.524, 0.524] rad
